In [ ]:
# Cài đặt thư viện cần thiết
!pip -q install --no-deps timm==0.4.12
!pip -q install --no-deps torchscale sentencepiece ftfy
!pip -q install -U open_clip_torch regex tqdm huggingface_hub
!pip -q install -U --no-deps "sentence-transformers[image]>=5.4.0"
!pip -q install -U --no-deps "transformers>=4.46.3" "accelerate" "einops" "safetensors"
!pip -q install tensorboardX torchmetrics fairscale

import os
from pathlib import Path

# 1. Clone BEiT-3
REPO_DIR = Path("/kaggle/working/unilm")
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/microsoft/unilm.git /kaggle/working/unilm

CKPT_DIR = Path("/kaggle/working/beit3_ckpt")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if not (CKPT_DIR / "beit3_base_patch16_384_coco_retrieval.pth").exists():
    !wget -q -nc -O /kaggle/working/beit3_ckpt/beit3_base_patch16_384_coco_retrieval.pth \
    https://github.com/addf400/files/releases/download/beit3/beit3_base_patch16_384_coco_retrieval.pth
if not (CKPT_DIR / "beit3.spm").exists():
    !wget -q -nc -O /kaggle/working/beit3_ckpt/beit3.spm \
    https://github.com/addf400/files/releases/download/beit3/beit3.spm

# 2. Clone Perception
PE_REPO_DIR = Path("/kaggle/working/perception_models")
if not PE_REPO_DIR.exists():
    !git clone --depth 1 https://github.com/facebookresearch/perception_models.git /kaggle/working/perception_models
!pip -q install -e /kaggle/working/perception_models --no-deps

print("Đã chuẩn bị xong môi trường!")

In [ ]:
import sys
from pathlib import Path

# 1. Khôi phục lại file utils.py gốc từ git để lấy lại hàm ClipLoss và các hàm khác
!cd /kaggle/working/unilm && git checkout beit3/utils.py

# 2. Patch file an toàn (Giống hệt cách đã hoạt động trong file aic-emd-beit-3 của bạn)
utils_path = Path("/kaggle/working/unilm/beit3/utils.py")
if utils_path.exists():
    text = utils_path.read_text(encoding="utf-8")
    
    # Fix torch._six
    text = text.replace(
        "from torch._six import inf", 
        "from math import inf"
    )
    
    # Fix tensorboardX 
    text = text.replace(
        "from tensorboardX import SummaryWriter",
        """try:
    from tensorboardX import SummaryWriter
except Exception:
    class SummaryWriter:
        def __init__(self, *args, **kwargs):
            pass
        def add_scalar(self, *args, **kwargs):
            pass
        def add_scalars(self, *args, **kwargs):
            pass
        def add_text(self, *args, **kwargs):
            pass
        def close(self):
            pass"""
    )
    utils_path.write_text(text, encoding="utf-8")

# 3. Dọn dẹp cache module để Python buộc phải đọc lại file utils.py vừa được sửa
for module_name in list(sys.modules.keys()):
    if module_name in ["utils", "modeling_finetune", "modeling_utils"]:
        del sys.modules[module_name]

print("Đã khôi phục và patch file utils.py thành công! Hàm ClipLoss đã quay trở lại.")

In [ ]:
import sys
import gc
import math
import heapq
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# IMPORT TRƯỚC ĐỂ TRÁNH XUNG ĐỘT (NAMESPACE COLLISION)
from sentence_transformers import SentenceTransformer
import datasets 
import open_clip

# Thêm đường dẫn mã nguồn vào sys.path
BEIT3_DIR = Path("/kaggle/working/unilm/beit3")
PE_REPO_DIR = Path("/kaggle/working/perception_models")

if str(BEIT3_DIR) not in sys.path:
    sys.path.append(str(BEIT3_DIR))
if str(PE_REPO_DIR) not in sys.path:
    sys.path.append(str(PE_REPO_DIR))

# Tiếp tục import của BEiT-3 và Perception
import timm
import modeling_finetune
import sentencepiece as spm
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}\n")

# ================================
# 1. LOAD BGE-VL-LARGE
# ================================
print("Loading BGE-VL-Large...")
bge_model = SentenceTransformer("BAAI/BGE-VL-large", trust_remote_code=True, device=device)

# ================================
# 2. LOAD METACLIP
# ================================
print("Loading MetaCLIP...")
metaclip_model, _, metaclip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-quickgelu", pretrained="metaclip_fullcc", device=device
)
metaclip_tokenizer = open_clip.get_tokenizer("ViT-L-14-quickgelu")
metaclip_model.eval()

# ================================
# 3. LOAD BEIT-3
# ================================
print("Loading BEiT-3...")
beit3_model = timm.create_model("beit3_base_patch16_384_retrieval", pretrained=False)
ckpt = torch.load("/kaggle/working/beit3_ckpt/beit3_base_patch16_384_coco_retrieval.pth", map_location="cpu", weights_only=False)
new_state_dict = {k[len("module."):]: v for k, v in ckpt["model"].items() if k.startswith("module.")}
beit3_model.load_state_dict(new_state_dict, strict=False)
beit3_model = beit3_model.to(device).eval()

sp = spm.SentencePieceProcessor()
sp.load("/kaggle/working/beit3_ckpt/beit3.spm")
BOS_ID, PAD_ID, EOS_ID = max(sp.piece_to_id("<s>"), 0), max(sp.piece_to_id("<pad>"), 1), max(sp.piece_to_id("</s>"), 2)

# ================================
# 4. LOAD PERCEPTION
# ================================
print("Loading Perception...")
perception_model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True).to(device).eval()
perception_tokenizer = pe_transforms.get_text_tokenizer(perception_model.context_length)

print("\nĐã load thành công toàn bộ 4 models!")

In [ ]:
@torch.no_grad()
def encode_bge_text(query):
    q_emb = bge_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    return q_emb.astype("float32")

@torch.no_grad()
def encode_metaclip_text(query):
    tokens = metaclip_tokenizer([query]).to(device)
    features = F.normalize(metaclip_model.encode_text(tokens), dim=-1)
    return features[0].detach().cpu().numpy().astype("float32")

@torch.no_grad()
def encode_beit3_text(query, max_len=64):
    ids = sp.encode(query, out_type=int)
    ids = [BOS_ID] + ids[:max_len - 2] + [EOS_ID]
    ids = ids + [PAD_ID] * (max_len - len(ids))
    
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)
    output = beit3_model(
        image=None, text_description=input_ids, 
        padding_mask=input_ids.eq(PAD_ID).to(device), only_infer=True
    )
    text_emb = output[1] if isinstance(output, tuple) else output.get("language_cls", output)
    return F.normalize(text_emb, dim=-1)[0].detach().cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_perception_text(query):
    text = perception_tokenizer([query]).to(device)
    if hasattr(perception_model, "encode_text"):
        features = perception_model.encode_text(text)
        if isinstance(features, tuple): features = features[0]
    else:
        h = w = perception_model.image_size if isinstance(perception_model.image_size, int) else perception_model.image_size[0]
        _, features, _ = perception_model(torch.zeros((1, 3, h, w), device=device), text)
        
    return F.normalize(features, dim=-1)[0].detach().cpu().numpy().astype("float32")

print("Đã định nghĩa xong các hàm Encode Text!")

In [ ]:
import math
import heapq
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


def search_embeddings(q_emb, emb_dir, file_prefix, top_k=100):
    """Search every embedding chunk and keep a global top-k heap."""
    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    heap, counter = [], 0
    emb_dir_path = Path(emb_dir)

    if not emb_dir_path.exists():
        print(f"Cảnh báo: Không tìm thấy thư mục {emb_dir}")
        return pd.DataFrame()

    emb_files = sorted(emb_dir_path.glob(f"{file_prefix}_*.npy"))
    if not emb_files:
        print(f"Cảnh báo: Không có file {file_prefix}_*.npy nào trong {emb_dir}")
        return pd.DataFrame()

    for emb_file in tqdm(emb_files, desc=f"Searching {file_prefix}"):
        chunk_id = emb_file.stem.split("_")[-1]
        meta_file = emb_dir_path / f"metadata_chunk_{chunk_id}.csv"

        if not meta_file.exists():
            print(f"Bỏ qua {emb_file.name}: thiếu {meta_file.name}")
            continue

        embs = np.load(emb_file, mmap_mode="r").astype("float32")
        metas = pd.read_csv(meta_file)

        if len(embs) != len(metas):
            print(
                f"Bỏ qua {emb_file.name}: số embedding ({len(embs)}) "
                f"khác metadata ({len(metas)})."
            )
            continue

        if len(embs) == 0:
            continue

        scores = embs @ q_emb
        k = min(top_k, len(scores))
        idxs = np.argpartition(-scores, kth=k - 1)[:k]

        for idx in idxs:
            item = (
                float(scores[idx]),
                counter,
                metas.iloc[int(idx)].to_dict(),
            )
            counter += 1

            if len(heap) < top_k:
                heapq.heappush(heap, item)
            elif item[0] > heap[0][0]:
                heapq.heapreplace(heap, item)

    results = sorted(heap, key=lambda item: item[0], reverse=True)
    rows = []

    for rank, (score, _, row) in enumerate(results, start=1):
        normalized_row = dict(row)
        normalized_row["rank"] = rank
        normalized_row["score"] = score
        rows.append(normalized_row)

    return pd.DataFrame(rows)


def compute_rrf(list_of_dfs, k=60):
    """Fuse result lists with Reciprocal Rank Fusion."""
    rrf_scores: dict[str, float] = {}
    metadata_dict: dict[str, dict[str, Any]] = {}

    for df in list_of_dfs:
        if df.empty:
            continue

        required_columns = {"image_path", "rank"}
        if not required_columns.issubset(df.columns):
            print(
                "Bỏ qua một result list vì thiếu cột: "
                f"{sorted(required_columns - set(df.columns))}"
            )
            continue

        for _, row in df.iterrows():
            image_path = str(row["image_path"])
            rank = int(row["rank"])

            if image_path not in rrf_scores:
                rrf_scores[image_path] = 0.0
                metadata_dict[image_path] = row.to_dict()

            rrf_scores[image_path] += 1.0 / (k + rank)

    final_df = pd.DataFrame(
        [
            {
                **metadata_dict[image_path],
                "rrf_score": score,
            }
            for image_path, score in rrf_scores.items()
        ]
    )

    if final_df.empty:
        return final_df

    final_df = (
        final_df
        .sort_values("rrf_score", ascending=False)
        .reset_index(drop=True)
    )
    final_df["final_rank"] = final_df.index + 1
    final_df["score"] = final_df["rrf_score"]
    return final_df


# Ví dụ hợp lệ:
# L21_V002_t00005.600_f00000168_s000001.jpg
FRAME_PATTERN = re.compile(
    r"^(?P<video_id>L\d{2}_V\d{3})"
    r"_t(?P<timestamp>\d+(?:\.\d+)?)"
    r"_f(?P<frame_idx>\d+)"
    r"_s(?P<shot_id>\d+)$",
    re.IGNORECASE,
)


def normalize_kaggle_path(raw_path: Any) -> Path:
    """
    Chuẩn hóa metadata path về đường dẫn local của Kaggle.

    Hỗ trợ:
    - /kaggle/input/...
    - https://www.kaggle.com/kaggle/input/...
    - https://www.kaggle.com/input/...
    """
    value = str(raw_path).strip()

    prefixes = (
        "https://www.kaggle.com/kaggle/input/",
        "https://www.kaggle.com/input/",
    )

    for prefix in prefixes:
        if value.startswith(prefix):
            value = "/kaggle/input/" + value[len(prefix):].lstrip("/")
            break

    return Path(value)


def parse_frame_filename(filename: str) -> dict[str, Any] | None:
    """
    Parse metadata trực tiếp từ tên frame.

    L21_V002_t00005.600_f00000168_s000001.jpg
      -> video_id=L21_V002
      -> timestamp_seconds=5.600
      -> frame_index=168
      -> shot_id=1
    """
    match = FRAME_PATTERN.fullmatch(Path(filename).stem)
    if match is None:
        return None

    return {
        "video_id": match.group("video_id").upper(),
        "timestamp_seconds": float(match.group("timestamp")),
        "frame_index": int(match.group("frame_idx")),
        "shot_id": int(match.group("shot_id")),
    }


def format_timestamp(seconds: float) -> str:
    """HH:MM:SS.mmm without losing the millisecond part."""
    total_milliseconds = round(max(0.0, float(seconds)) * 1000)
    hours, remainder = divmod(total_milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    whole_seconds, milliseconds = divmod(remainder, 1000)

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{whole_seconds:02d}."
        f"{milliseconds:03d}"
    )


def _first_non_null(row: pd.Series, *keys: str):
    for key in keys:
        if key not in row:
            continue

        value = row[key]
        if pd.notna(value) and str(value).strip() != "":
            return value

    return None


def fix_results_metadata(results: pd.DataFrame) -> pd.DataFrame:
    """
    Chuẩn hóa mỗi retrieval row bằng metadata trong tên frame.

    Tên frame là source of truth. Metadata CSV chỉ được dùng làm fallback
    nếu filename không đúng pattern.
    """
    if results.empty:
        return results.copy()

    normalized_rows: list[dict[str, Any]] = []

    for _, row in results.iterrows():
        local_path = normalize_kaggle_path(row.get("image_path", ""))
        filename = local_path.name
        parsed = parse_frame_filename(filename)

        if parsed is not None:
            video_id = parsed["video_id"]
            timestamp_seconds = parsed["timestamp_seconds"]
            frame_index = parsed["frame_index"]
            shot_id = parsed["shot_id"]
            metadata_source = "filename"
        else:
            video_id_raw = _first_non_null(
                row,
                "video_id",
                "video_id_fixed",
                "video_title",
            )
            timestamp_raw = _first_non_null(
                row,
                "timestamp_seconds",
                "timestamp",
                "timestamp_fixed",
            )
            frame_raw = _first_non_null(
                row,
                "frame_index",
                "frame_idx",
                "frame_idx_fixed",
            )
            shot_raw = _first_non_null(
                row,
                "shot_id",
                "shot_id_fixed",
            )

            video_id = (
                str(video_id_raw).upper()
                if video_id_raw is not None
                else None
            )
            timestamp_seconds = (
                float(timestamp_raw)
                if timestamp_raw is not None
                else None
            )
            frame_index = (
                int(frame_raw)
                if frame_raw is not None
                else None
            )
            shot_id = (
                int(shot_raw)
                if shot_raw is not None
                else None
            )
            metadata_source = "csv_fallback"

        normalized = row.to_dict()
        normalized.update(
            {
                "local_image_path": str(local_path),
                "filename": filename,
                "video_id_fixed": video_id,
                "video_title": video_id,
                "timestamp_fixed": timestamp_seconds,
                "timestamp_seconds": timestamp_seconds,
                "time_fixed": (
                    format_timestamp(timestamp_seconds)
                    if timestamp_seconds is not None
                    else ""
                ),
                "frame_idx_fixed": frame_index,
                "frame_index": frame_index,
                "shot_id_fixed": shot_id,
                "image_exists": local_path.is_file(),
                "metadata_source": metadata_source,
                "metadata_valid": (
                    video_id is not None
                    and timestamp_seconds is not None
                    and frame_index is not None
                ),
            }
        )
        normalized_rows.append(normalized)

    return pd.DataFrame(normalized_rows)


def diversify_results(
    results: pd.DataFrame,
    min_gap_seconds=5.0,
    max_per_video=5,
    max_results=20,
):
    """Keep temporally diverse frames and discard rows with invalid metadata."""
    if results.empty:
        return results.copy()

    selected: list[pd.Series] = []
    selected_times: dict[str, list[float]] = {}

    ordered = results.sort_values("score", ascending=False).reset_index(drop=True)

    for _, row in ordered.iterrows():
        video_id = row.get("video_id_fixed")
        timestamp = row.get("timestamp_fixed")

        if (
            video_id is None
            or pd.isna(video_id)
            or timestamp is None
            or pd.isna(timestamp)
        ):
            continue

        video_id = str(video_id).upper()
        timestamp = float(timestamp)
        selected_times.setdefault(video_id, [])

        if len(selected_times[video_id]) >= max_per_video:
            continue

        if any(
            abs(timestamp - previous) < min_gap_seconds
            for previous in selected_times[video_id]
        ):
            continue

        selected_times[video_id].append(timestamp)
        selected.append(row)

        if len(selected) >= max_results:
            break

    if not selected:
        return pd.DataFrame(columns=results.columns)

    output = pd.DataFrame(selected).reset_index(drop=True)
    output["final_rank"] = range(1, len(output) + 1)
    return output


def visualize_results(results, n=12, cols=4):
    if results.empty:
        print("Lỗi: DataFrame rỗng, không có gì để hiển thị.")
        return

    valid_rows = [
        row
        for _, row in results.iterrows()
        if Path(str(row["local_image_path"])).is_file()
    ][:n]

    if not valid_rows:
        print("Lỗi: Không tìm thấy file ảnh gốc trên ổ cứng.")
        return

    rows = math.ceil(len(valid_rows) / cols)
    plt.figure(figsize=(cols * 4.5, rows * 3.6))

    for index, row in enumerate(valid_rows):
        plt.subplot(rows, cols, index + 1)
        plt.imshow(
            Image.open(row["local_image_path"]).convert("RGB")
        )
        plt.axis("off")
        plt.title(
            (
                f"Rank {int(row['final_rank'])} | "
                f"RRF={row['rrf_score']:.3f}\n"
                f"{row['video_title']} | {row['time_fixed']}"
            ),
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()


# Sanity check đúng case người dùng yêu cầu.
_example = parse_frame_filename(
    "L21_V002_t00005.600_f00000168_s000001.jpg"
)
assert _example == {
    "video_id": "L21_V002",
    "timestamp_seconds": 5.6,
    "frame_index": 168,
    "shot_id": 1,
}

print("Đã khởi tạo xong retrieval helpers và frame metadata parser!")


In [ ]:
# ============================================================
# CẬP NHẬT ĐƯỜNG DẪN ĐẾN THƯ MỤC CHỨA ĐẶC TRƯNG .npy CỦA BẠN
# ============================================================
BGE_DIR        = "/kaggle/input/notebooks/thnhtrnhvn/aic-emd/embeddings_bge_vl_large"
METACLIP_DIR   = "/kaggle/input/notebooks/thnhtrnhvn/aic-emd-metaclip2/embeddings_metaclip_openclip"
BEIT3_DIR      = "/kaggle/input/notebooks/thnhtrnhvn/aic-emd-beit-3/embeddings_beit3_base_coco_retrieval"
PERCEPTION_DIR = "/kaggle/input/notebooks/thnhtrnhvn/aic-2026-emd-perception/embeddings_perception_pe_core_l14_336"

query = "three people (two women and one man) sitting together, focusing on playing a round, hollow metal instrument with indentations to produce sound when tapped. One person wearing a white shirt is sitting between the two people wearing black shirts. In the background is a multi-tiered bookshelf filled with books of various colors."
TOP_K_SEARCH = 100

print(f"Xử lý Query: '{query}'")

# 1. Encode query
q_bge = encode_bge_text(query)
q_metaclip = encode_metaclip_text(query)
q_beit3 = encode_beit3_text(query)
q_perception = encode_perception_text(query)

# 2. Search từng model
print("\nĐang quét dữ liệu BGE...")
df_bge = search_embeddings(q_bge, BGE_DIR, "bge_image_emb_chunk", top_k=TOP_K_SEARCH)

print("\nĐang quét dữ liệu MetaCLIP...")
df_metaclip = search_embeddings(q_metaclip, METACLIP_DIR, "metaclip_image_emb_chunk", top_k=TOP_K_SEARCH)

print("\nĐang quét dữ liệu BEiT-3...")
df_beit3 = search_embeddings(q_beit3, BEIT3_DIR, "beit3_image_emb_chunk", top_k=TOP_K_SEARCH)

print("\nĐang quét dữ liệu Perception...")
df_perception = search_embeddings(q_perception, PERCEPTION_DIR, "perception_image_emb_chunk", top_k=TOP_K_SEARCH)

# 3. Tính RRF
print("\nÁp dụng Reciprocal Rank Fusion (RRF)...")
rrf_results = compute_rrf([df_bge, df_metaclip, df_beit3, df_perception], k=60)

# 4. Sửa Metadata và Diversify (lọc trùng)
if not rrf_results.empty:
    fixed_results = fix_results_metadata(rrf_results)
    final_results = diversify_results(fixed_results, min_gap_seconds=5.0, max_per_video=5, max_results=20)

    print(f"\nHoàn thành! Giữ lại {len(final_results)} kết quả (đã lọc trùng).")
    
    # Hiển thị bảng tóm tắt
    display_cols = ["final_rank", "rrf_score", "video_id_fixed", "time_fixed", "frame_idx_fixed", "filename"]
    display(final_results[[col for col in display_cols if col in final_results.columns]].head(10))
    
    # Vẽ ảnh
    visualize_results(final_results, n=12)
    
    output_path = "/kaggle/working/search_results_rrf_final.csv"
    final_results.to_csv(output_path, index=False)
    print(f"Đã lưu kết quả tại: {output_path}")
else:
    print("Pipeline thất bại do không có kết quả từ các model. Vui lòng kiểm tra lại đường dẫn .npy!")

In [ ]:
# ============================================================
# FASTAPI BACKEND: SEARCH + FRAME -> ORIGINAL VIDEO PLAYBACK
# Replace the whole old backend/ngrok cell with this cell.
# ============================================================

!pip install -q -U fastapi uvicorn pyngrok nest_asyncio python-multipart pydantic

import mimetypes
import os
import re
from pathlib import Path
from time import perf_counter
from typing import Iterator
from uuid import uuid4

import nest_asyncio
import pandas as pd
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, Response, StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from pyngrok import ngrok


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
DEFAULT_VIDEO_ROOT = Path("/kaggle/input/aic2025")

VIDEO_EXTENSIONS = {
    ".mp4",
    ".m4v",
    ".mov",
    ".webm",
    ".mkv",
    ".avi",
}

# Lower number = preferred when duplicate video IDs exist.
VIDEO_EXTENSION_PRIORITY = {
    ".mp4": 0,
    ".m4v": 1,
    ".webm": 2,
    ".mov": 3,
    ".mkv": 4,
    ".avi": 5,
}

BROWSER_FRIENDLY_VIDEO_EXTENSIONS = {
    ".mp4",
    ".m4v",
    ".webm",
    ".mov",
}

VIDEO_ID_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?P<video_id>L\d{2}_V\d{3})(?![A-Z0-9])",
    re.IGNORECASE,
)

STREAM_CHUNK_SIZE = 1024 * 1024


def candidate_video_roots() -> list[Path]:
    """
    Resolve dataset roots without hard-coding the internal video folder shape.

    Primary expected dataset:
        /kaggle/input/aic2025

    Override:
        environment/Kaggle Secret AIC_VIDEO_ROOT
    """
    raw_roots: list[Path] = []

    configured_root = os.getenv("AIC_VIDEO_ROOT", "").strip()
    if configured_root:
        raw_roots.append(Path(configured_root))

    raw_roots.append(DEFAULT_VIDEO_ROOT)

    # Fallback: tolerate Kaggle datasets whose mounted slug differs slightly.
    if KAGGLE_INPUT_ROOT.exists():
        for child in KAGGLE_INPUT_ROOT.iterdir():
            if not child.is_dir():
                continue

            normalized_name = child.name.lower().replace("_", "-")
            if "aic2025" in normalized_name or "aic-2025" in normalized_name:
                raw_roots.append(child)

    unique_roots: list[Path] = []
    seen: set[str] = set()

    for root in raw_roots:
        try:
            key = str(root.resolve())
        except OSError:
            key = str(root)

        if key not in seen:
            seen.add(key)
            unique_roots.append(root)

    return unique_roots


def video_path_preference(path: Path) -> tuple[int, int, int]:
    """Prefer browser-friendly codecs/containers and then shorter paths."""
    extension_rank = VIDEO_EXTENSION_PRIORITY.get(
        path.suffix.lower(),
        99,
    )
    video_folder_bonus = 0 if "video" in str(path.parent).lower() else 1
    return (
        extension_rank,
        video_folder_bonus,
        len(str(path)),
    )


def build_video_index() -> tuple[dict[str, Path], list[Path]]:
    """
    Scan once and map:
        L21_V002 -> /kaggle/input/aic2025/.../L21_V002.mp4
    """
    index: dict[str, Path] = {}
    scanned_roots: list[Path] = []

    for root in candidate_video_roots():
        if not root.is_dir():
            continue

        scanned_roots.append(root)

        for path in root.rglob("*"):
            if (
                not path.is_file()
                or path.suffix.lower() not in VIDEO_EXTENSIONS
            ):
                continue

            match = VIDEO_ID_PATTERN.search(path.stem)
            if match is None:
                continue

            video_id = match.group("video_id").upper()
            current = index.get(video_id)

            if (
                current is None
                or video_path_preference(path)
                < video_path_preference(current)
            ):
                index[video_id] = path

    return index, scanned_roots


VIDEO_INDEX, VIDEO_ROOTS_SCANNED = build_video_index()


def rebuild_video_index() -> None:
    global VIDEO_INDEX, VIDEO_ROOTS_SCANNED
    VIDEO_INDEX, VIDEO_ROOTS_SCANNED = build_video_index()


def resolve_video(video_id: str) -> Path | None:
    normalized_video_id = video_id.strip().upper()

    if VIDEO_ID_PATTERN.fullmatch(normalized_video_id) is None:
        return None

    path = VIDEO_INDEX.get(normalized_video_id)
    return path if path is not None and path.is_file() else None


# ------------------------------------------------------------
# 2. HTTP RANGE STREAMING
# ------------------------------------------------------------

def parse_range_header(
    range_header: str,
    file_size: int,
) -> tuple[int, int]:
    """
    Parse one HTTP byte range.

    Supported:
        bytes=0-999
        bytes=1000-
        bytes=-500

    Multiple ranges are deliberately rejected because the HTML <video>
    element only needs normal single-range responses for seeking.
    """
    if file_size <= 0:
        raise ValueError("Video file is empty.")

    normalized = range_header.strip()

    if "," in normalized:
        raise ValueError("Multiple byte ranges are not supported.")

    match = re.fullmatch(
        r"bytes=(?P<start>\d*)-(?P<end>\d*)",
        normalized,
    )
    if match is None:
        raise ValueError("Invalid Range header.")

    start_raw = match.group("start")
    end_raw = match.group("end")

    if start_raw == "" and end_raw == "":
        raise ValueError("Empty Range header.")

    if start_raw == "":
        suffix_length = int(end_raw)
        if suffix_length <= 0:
            raise ValueError("Invalid suffix byte range.")

        start = max(file_size - suffix_length, 0)
        end = file_size - 1
    else:
        start = int(start_raw)
        end = int(end_raw) if end_raw else file_size - 1

    if start >= file_size:
        raise ValueError("Requested range starts after the file end.")

    end = min(end, file_size - 1)

    if start < 0 or end < start:
        raise ValueError("Requested range is not satisfiable.")

    return start, end


def iter_file_range(
    path: Path,
    start: int,
    end: int,
    chunk_size: int = STREAM_CHUNK_SIZE,
) -> Iterator[bytes]:
    with path.open("rb") as file:
        file.seek(start)
        remaining = end - start + 1

        while remaining > 0:
            chunk = file.read(min(chunk_size, remaining))
            if not chunk:
                break

            remaining -= len(chunk)
            yield chunk


def video_media_type(path: Path) -> str:
    guessed, _ = mimetypes.guess_type(path.name)

    if guessed:
        return guessed

    fallback_by_extension = {
        ".mkv": "video/x-matroska",
        ".m4v": "video/x-m4v",
    }
    return fallback_by_extension.get(
        path.suffix.lower(),
        "application/octet-stream",
    )


def stream_video_response(
    video_path: Path,
    request: Request,
):
    file_size = video_path.stat().st_size
    media_type = video_media_type(video_path)
    safe_filename = video_path.name.replace('"', "")
    range_header = request.headers.get("range")

    common_headers = {
        "Accept-Ranges": "bytes",
        "Cache-Control": "public, max-age=3600",
        "Content-Disposition": f'inline; filename="{safe_filename}"',
    }

    if request.method == "HEAD":
        return Response(
            status_code=200,
            media_type=media_type,
            headers={
                **common_headers,
                "Content-Length": str(file_size),
            },
        )

    if not range_header:
        return FileResponse(
            path=str(video_path),
            media_type=media_type,
            headers=common_headers,
        )

    try:
        start, end = parse_range_header(
            range_header,
            file_size,
        )
    except ValueError as error:
        raise HTTPException(
            status_code=416,
            detail=str(error),
            headers={
                "Accept-Ranges": "bytes",
                "Content-Range": f"bytes */{file_size}",
            },
        ) from error

    content_length = end - start + 1

    return StreamingResponse(
        iter_file_range(video_path, start, end),
        status_code=206,
        media_type=media_type,
        headers={
            **common_headers,
            "Content-Range": (
                f"bytes {start}-{end}/{file_size}"
            ),
            "Content-Length": str(content_length),
        },
    )


# ------------------------------------------------------------
# 3. REQUEST CONTRACT
# ------------------------------------------------------------

class SearchSceneRequest(BaseModel):
    scene_id: int = Field(default=1, ge=1)
    visual_query: str = ""
    ocr_query: str = ""
    asr_query: str = ""


class SearchOptions(BaseModel):
    top_k_per_modality: int = Field(
        default=100,
        ge=1,
        le=1000,
    )
    display_result_count: int = Field(
        default=20,
        ge=1,
        le=500,
    )


class SearchRequest(BaseModel):
    """
    Compatible with both:
      1) old notebook request:
         {"query": "...", "top_k": 100}

      2) current React request:
         {
           "task": "task_1",
           "scenes": [...],
           "options": {...}
         }
    """
    query: str | None = None
    top_k: int | None = Field(
        default=None,
        ge=1,
        le=1000,
    )

    task: str = "task_1"
    feedback: str = ""
    scenes: list[SearchSceneRequest] = Field(
        default_factory=list
    )
    options: SearchOptions = Field(
        default_factory=SearchOptions
    )

    def resolved_query(self) -> str:
        if self.query and self.query.strip():
            return self.query.strip()

        query_parts: list[str] = []

        for scene in self.scenes:
            for value in (
                scene.visual_query,
                scene.ocr_query,
                scene.asr_query,
            ):
                normalized = value.strip()
                if normalized:
                    query_parts.append(normalized)

        return " ".join(query_parts).strip()

    def resolved_top_k(self) -> int:
        return self.top_k or self.options.top_k_per_modality

    def resolved_result_count(self) -> int:
        if self.query is not None and not self.scenes:
            return 20

        return self.options.display_result_count


# ------------------------------------------------------------
# 4. FASTAPI APP
# ------------------------------------------------------------

app = FastAPI(
    title="AIC Retrieval API",
    version="3.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=[
        "Accept-Ranges",
        "Content-Range",
        "Content-Length",
    ],
)

if not KAGGLE_INPUT_ROOT.is_dir():
    raise RuntimeError(
        "Missing /kaggle/input. Run this backend inside Kaggle."
    )

app.mount(
    "/images",
    StaticFiles(directory=str(KAGGLE_INPUT_ROOT)),
    name="images",
)


def image_public_url(
    request: Request,
    local_path: str | Path,
) -> str | None:
    """
    Return an absolute URL only when the frame is an existing file
    inside /kaggle/input. This prevents bad paths and traversal.
    """
    candidate = Path(local_path)

    try:
        resolved_candidate = candidate.resolve(strict=True)
        resolved_root = KAGGLE_INPUT_ROOT.resolve(strict=True)
        relative = resolved_candidate.relative_to(resolved_root)
    except (FileNotFoundError, ValueError, OSError):
        return None

    return str(
        request.url_for(
            "images",
            path=relative.as_posix(),
        )
    )


def format_timestamp_display(seconds: float) -> str:
    # Reuse the helper from the retrieval cell when available.
    return format_timestamp(seconds)


@app.get("/api/v1/health")
@app.get("/api/health")
async def health():
    return {
        "status": "ok",
        "service": "aic-retrieval-kaggle",
        "indexed_videos": len(VIDEO_INDEX),
        "video_roots_scanned": [
            str(path)
            for path in VIDEO_ROOTS_SCANNED
        ],
        "sample_video_ids": sorted(VIDEO_INDEX)[:10],
    }


@app.post("/api/v1/videos/reindex")
async def reindex_videos():
    rebuild_video_index()

    return {
        "status": "ok",
        "indexed_videos": len(VIDEO_INDEX),
        "video_roots_scanned": [
            str(path)
            for path in VIDEO_ROOTS_SCANNED
        ],
    }


@app.get("/api/v1/videos/{video_id}/info")
@app.get("/api/videos/{video_id}/info")
async def get_video_info(video_id: str):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=(
                f"Video {normalized_video_id} was not found. "
                "Check that the dataset is mounted at "
                "/kaggle/input/aic2025 or set AIC_VIDEO_ROOT, "
                "then call POST /api/v1/videos/reindex."
            ),
        )

    return {
        "video_id": normalized_video_id,
        "filename": video_path.name,
        "extension": video_path.suffix.lower(),
        "size_bytes": video_path.stat().st_size,
        "browser_friendly": (
            video_path.suffix.lower()
            in BROWSER_FRIENDLY_VIDEO_EXTENSIONS
        ),
        # Do not expose the server's absolute local filesystem path.
        "video_url": f"/api/v1/videos/{normalized_video_id}/stream",
    }


@app.api_route(
    "/api/v1/videos/{video_id}/stream",
    methods=["GET", "HEAD"],
    name="stream_video",
)
async def stream_video(
    video_id: str,
    request: Request,
):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=(
                f"Video {normalized_video_id} was not found. "
                "Check AIC_VIDEO_ROOT and the original video filename."
            ),
        )

    return stream_video_response(
        video_path,
        request,
    )


# Legacy alias kept for the old notebook/UI contract.
@app.api_route(
    "/api/videos/{video_id}",
    methods=["GET", "HEAD"],
    name="stream_video_legacy",
)
async def stream_video_legacy(
    video_id: str,
    request: Request,
):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=f"Video {normalized_video_id} was not found.",
        )

    return stream_video_response(
        video_path,
        request,
    )


@app.get("/api/v1/videos/{video_id}/clip")
async def get_video_clip(
    video_id: str,
    center_time: float,
    before: float = 5.0,
    after: float = 5.0,
):
    """
    Compatibility endpoint for the existing VideoPlayerModal.
    The updated frontend below no longer needs this extra request,
    but keeping it avoids breaking older builds.
    """
    if center_time < 0 or before < 0 or after < 0:
        raise HTTPException(
            status_code=422,
            detail="center_time, before and after must be non-negative.",
        )

    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)
    video_available = video_path is not None

    return {
        "video_id": normalized_video_id,
        "center_time": center_time,
        "start_time": max(0.0, center_time - before),
        "end_time": center_time + after,
        "video_available": video_available,
        "video_url": (
            f"/api/v1/videos/{normalized_video_id}/stream"
            if video_available
            else None
        ),
        "poster_url": None,
    }


# ------------------------------------------------------------
# 5. SEARCH ENDPOINT
# ------------------------------------------------------------

@app.post(
    "/api/v1/search",
    name="search_video_v1",
)
@app.post(
    "/api/search",
    name="search_video_legacy",
)
async def search_video(
    req: SearchRequest,
    request: Request,
):
    started_at = perf_counter()
    query = req.resolved_query()

    if not query:
        raise HTTPException(
            status_code=422,
            detail=(
                "Provide either query or at least one non-empty "
                "visual_query, ocr_query or asr_query."
            ),
        )

    top_k = req.resolved_top_k()
    display_result_count = req.resolved_result_count()

    print(f"\n[API] Query: {query!r}")
    print(
        f"[API] top_k={top_k}, "
        f"display_result_count={display_result_count}"
    )

    # BƯỚC 1: Encode query.
    q_bge = encode_bge_text(query)
    q_metaclip = encode_metaclip_text(query)
    q_beit3 = encode_beit3_text(query)
    q_perception = encode_perception_text(query)

    # BƯỚC 2: Search every embedding store.
    df_bge = search_embeddings(
        q_bge,
        BGE_DIR,
        "bge_image_emb_chunk",
        top_k=top_k,
    )
    df_metaclip = search_embeddings(
        q_metaclip,
        METACLIP_DIR,
        "metaclip_image_emb_chunk",
        top_k=top_k,
    )
    df_beit3 = search_embeddings(
        q_beit3,
        BEIT3_DIR,
        "beit3_image_emb_chunk",
        top_k=top_k,
    )
    df_perception = search_embeddings(
        q_perception,
        PERCEPTION_DIR,
        "perception_image_emb_chunk",
        top_k=top_k,
    )

    # BƯỚC 3: RRF + metadata normalization + temporal diversity.
    rrf_results = compute_rrf(
        [
            df_bge,
            df_metaclip,
            df_beit3,
            df_perception,
        ],
        k=60,
    )

    warnings: list[dict[str, str]] = []
    results_json: list[dict] = []

    if not rrf_results.empty:
        fixed_results = fix_results_metadata(rrf_results)

        invalid_metadata_count = int(
            (~fixed_results["metadata_valid"]).sum()
        )
        if invalid_metadata_count:
            warnings.append(
                {
                    "module": "metadata",
                    "code": "INVALID_FRAME_FILENAME",
                    "message": (
                        f"Skipped {invalid_metadata_count} result(s) "
                        "whose frame metadata could not be parsed."
                    ),
                }
            )

        diversified_results = diversify_results(
            fixed_results,
            min_gap_seconds=5.0,
            max_per_video=5,
            max_results=display_result_count,
        )

        missing_frame_count = 0
        missing_video_ids: set[str] = set()

        for _, row in diversified_results.iterrows():
            frame_url = image_public_url(
                request,
                row["local_image_path"],
            )

            if frame_url is None:
                missing_frame_count += 1
                continue

            video_id = str(
                row["video_id_fixed"]
            ).upper()
            timestamp_seconds = float(
                row["timestamp_fixed"]
            )
            frame_index = int(
                row["frame_idx_fixed"]
            )
            video_path = resolve_video(video_id)
            video_available = video_path is not None

            if not video_available:
                missing_video_ids.add(video_id)

            video_url = (
                str(
                    request.url_for(
                        "stream_video",
                        video_id=video_id,
                    )
                )
                if video_available
                else None
            )

            rank = len(results_json) + 1
            timestamp_display = format_timestamp_display(
                timestamp_seconds
            )
            score = float(row["rrf_score"])

            # Includes both the current frontend contract and aliases
            # used by the old notebook frontend.
            results_json.append(
                {
                    "rank": rank,
                    "final_rank": rank,
                    "video_id": video_id,
                    "frame_index": frame_index,
                    "frame_idx": frame_index,
                    "timestamp_seconds": timestamp_seconds,
                    "timestamp_display": timestamp_display,
                    "time": timestamp_display,
                    "thumbnail_url": frame_url,
                    "frame_url": frame_url,
                    "image_url": frame_url,
                    "video_url": video_url,
                    "video_available": video_available,
                    "score": score,
                    "matched_scene": 1,
                    "modality_scores": {
                        "visual": score,
                        "ocr": None,
                        "asr": None,
                    },
                }
            )

        if missing_frame_count:
            warnings.append(
                {
                    "module": "frames",
                    "code": "FRAME_PATH_NOT_FOUND",
                    "message": (
                        f"Skipped {missing_frame_count} frame(s) "
                        "because the local path was missing or outside "
                        "/kaggle/input."
                    ),
                }
            )

        if missing_video_ids:
            preview = ", ".join(
                sorted(missing_video_ids)[:10]
            )
            suffix = (
                "..."
                if len(missing_video_ids) > 10
                else ""
            )
            warnings.append(
                {
                    "module": "videos",
                    "code": "VIDEO_NOT_FOUND",
                    "message": (
                        "No original video was indexed for: "
                        f"{preview}{suffix}. "
                        "Check /kaggle/input/aic2025 or AIC_VIDEO_ROOT."
                    ),
                }
            )

    processing_time_ms = round(
        (perf_counter() - started_at) * 1000
    )

    payload = {
        "status": "success",
        "search_id": str(uuid4()),
        "task": req.task,
        "processing_time_ms": processing_time_ms,
        "total_results": len(results_json),
        "indexed_videos": len(VIDEO_INDEX),
        "warnings": warnings,
        "results": results_json,
        # Backward-compatible alias for the old notebook UI.
        "data": results_json,
    }

    print(
        f"[API] Returned {len(results_json)} result(s); "
        f"indexed_videos={len(VIDEO_INDEX)}."
    )
    return payload


# ------------------------------------------------------------
# 6. NGROK + UVICORN
# ------------------------------------------------------------

def read_kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        value = os.getenv(name)
        return value.strip() if value else None


print("\n" + "=" * 72)
print(
    "VIDEO ROOTS SCANNED:",
    [str(path) for path in VIDEO_ROOTS_SCANNED],
)
print("INDEXED VIDEOS:", len(VIDEO_INDEX))
print("SAMPLE VIDEO IDS:", sorted(VIDEO_INDEX)[:10])

sample_id = "L21_V002"
sample_path = resolve_video(sample_id)
print(
    f"SAMPLE {sample_id}:",
    str(sample_path) if sample_path else "NOT FOUND",
)
print("=" * 72 + "\n")


NGROK_AUTH_TOKEN = read_kaggle_secret(
    "NGROK_AUTH_TOKEN"
)
NGROK_DOMAIN = read_kaggle_secret(
    "NGROK_DOMAIN"
)

public_url: str | None = None

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()

    connect_kwargs = {"addr": 8000}
    if NGROK_DOMAIN:
        connect_kwargs["domain"] = NGROK_DOMAIN

    tunnel = ngrok.connect(**connect_kwargs)
    public_url = tunnel.public_url

    print("\n" + "=" * 72)
    print("PUBLIC API:", public_url)
    print("FRONTEND .env:")
    print(
        f"VITE_API_BASE_URL={public_url}/api/v1"
    )
    print(
        "HEALTH:",
        f"{public_url}/api/v1/health",
    )
    print("=" * 72 + "\n")
else:
    print(
        "NGROK_AUTH_TOKEN is missing. "
        "The API will only listen on Kaggle localhost:8000."
    )


nest_asyncio.apply()
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    proxy_headers=True,
    forwarded_allow_ips="*",
)
server = uvicorn.Server(config)
await server.serve()
